# Study 2 justification coding

This notebook carries out the registered justification coding protocol for Modeling Guilty Minds, Study 2 (preregistration, section "Justification coding protocol"). It is Stage 2 of the Study 2 Analysis Plan and runs before the analysis script is frozen. It works in Google Colab or in a local Jupyter session, from the `study2_analysis` folder.

The notebook has four parts, run in order. Part A builds the coding prompt from the codebook, the decision rules and fourteen examples taken from Study 1 justifications outside the check set, sends one test request to confirm that the model accepts the registered settings, and freezes the protocol. Part B scores the check set: the language model codes the 100 check-set justifications once, with the frozen prompt, and the notebook computes Cohen's kappa against the hand codes. Part C codes every Study 1 and Study 2 justification with the language model, runs the inference-model cross-check, and writes `data/processed/justification_codes.csv`, the file the analysis script reads. Part D rebuilds that file from the archived raw outputs alone and confirms that it is identical, which shows that anyone with the archive can reproduce the codes without calling either model.

Every response from the language model is saved as it arrives, in `protocol/llm_raw/`, and every inference-model result in `protocol/nli_raw/`. Running a cell again reuses what is saved and sends only what is missing, so the notebook can be stopped and restarted (after a restart, run it again from the top), and running it after the coding is finished makes no new requests. A model at temperature 0 can still answer differently on a second request, so the archived outputs, not a new run, are the record of the coding.

The notebook never prints a Study 2 justification, a Study 2 code or a count of Study 2 codes. The Study 2 files it writes (`protocol/llm_raw/study2.jsonl`, `protocol/nli_raw/study2.csv` and `data/processed/justification_codes.csv`) stay closed until the analysis script is frozen.

The API key is never typed into the notebook. In Colab, add it once under Secrets (the key icon in the left sidebar) with the name `MPC_Claude_Key`, and allow this notebook to read it. In a local session, set the environment variable `MPC_Claude_Key` before starting Jupyter. If neither is found, the notebook asks for the key in a hidden field and keeps it only in memory.

## Setup

In Colab, choose a GPU runtime (Runtime, then Change runtime type) so that the inference model takes minutes rather than hours, and set `FOLDER_IN_DRIVE` to the folder's location in your Google Drive. The notebook needs only `data/study1_trials.csv`, `data/study1_names.csv`, `data/processed/study2_trials.csv` and the `protocol` folder, so the raw Pavlovia files and the Prolific export can stay off Google Drive. In a local session, start Jupyter from the `study2_analysis` folder.

In [2]:
# Where the study2_analysis folder is.
FOLDER_IN_DRIVE = "/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Mens Rea /Analysis/Mens Rea/study2_analysis"   # Colab: the folder's path inside your Google Drive
LOCAL_FOLDER = None                          # local: None means the folder Jupyter was started in

import os, sys, importlib.util, importlib.metadata
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = os.path.join("/content/drive", FOLDER_IN_DRIVE)
else:
    ROOT = LOCAL_FOLDER or os.getcwd()
os.chdir(ROOT)
needed = ["data/study1_trials.csv", "data/study1_names.csv", "protocol/check_set_key.csv",
          "protocol/check_set_hand_codes.csv", "protocol/coding_prompt_template.txt", "protocol/coding_examples.csv"]
missing = [f for f in needed if not os.path.exists(f)]
assert not missing, f"Not found in {ROOT}: {missing}. Is this the study2_analysis folder?"
print("Working in", ROOT, "(Colab)" if IN_COLAB else "(local)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Mens Rea /Analysis/Mens Rea/study2_analysis (Colab)


In [3]:
# The anthropic package is pinned to the version the protocol is frozen with. torch and transformers
# run the inference model; Colab has them already.
import subprocess
ANTHROPIC_VERSION = "1.12.1"
if importlib.util.find_spec("anthropic") is None or importlib.metadata.version("anthropic") != ANTHROPIC_VERSION:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"anthropic=={ANTHROPIC_VERSION}"], check=True)
import anthropic
assert anthropic.__version__ == ANTHROPIC_VERSION, (
    f"anthropic {anthropic.__version__} is loaded. Restart the session (Colab: Runtime, then Restart session) and run again.")
missing = [m for m in ("torch", "transformers", "numpy", "pandas") if importlib.util.find_spec(m) is None]
assert not missing, "Install these first: pip install " + " ".join(missing)
import json, hashlib, threading, zipfile, datetime, getpass, platform
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd, torch, transformers
print(f"Python {platform.python_version()} | anthropic {anthropic.__version__} | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

Python 3.13.15 | anthropic 1.12.1 | torch 2.11.0+cu130 | transformers 5.18.0 | pandas 2.2.3 | numpy 2.1.3


## Protocol settings

These are the registered settings, in one place. The freeze in Part A records them, and Parts B and C stop if any of them, the prompt, the examples or the hand codes differ from the frozen version.

The language model is Claude Opus 4.6 (`claude-opus-4-6`). The registration asks for a pinned model version run at temperature 0. Anthropic's current models (Opus 4.7 and later, Sonnet 5.5, Haiku 5.5) reject any temperature setting, so the protocol uses the most capable model that still accepts temperature 0. Its identifier names one fixed snapshot: from the 4.6 generation on, Anthropic's model identifiers carry no date, and the model behind an identifier never changes (Anthropic, "Model IDs and versioning").

In [4]:
# Language model (automated coder 2, the analytic codes)
MODEL = "claude-opus-4-6"
TEMPERATURE = 0
MAX_TOKENS = 50
CODES = ["Knew", "Suspected", "Unaware, at fault", "Unaware, no fault",
         "Act or situation only", "Evidentiary", "Uncodable"]
SCHEMA = {"type": "object", "properties": {"code": {"type": "string", "enum": CODES}},
          "required": ["code"], "additionalProperties": False}
USER_TEMPLATE = "<defendant>{name}</defendant>\n<justification>{text}</justification>"
TEXT_PREP = "Leading and trailing white space is removed from each justification; nothing else is changed."
INVALID_RULE = ("A response that cannot be read as one of the seven codes (a refusal, an answer cut off "
                "at the token limit, or a label outside the list) is coded Uncodable and counted.")

# Inference model (automated coder 1, the cross-check on the awareness-denied code)
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
NLI_REVISION = "b3546ea6b0346eb6f8d5d68b13c7dc6d0376b3d7"
NLI_PREMISE = "{name} is the defendant. {text}"
NLI_HYPOTHESES = ["{name} did not know that the package contained illicit material",
                  "{name} did not suspect that the package might contain illicit material"]
NLI_RULE = "Awareness denied when entailment is the most probable label for either hypothesis."

# Running (these do not change any result)
WORKERS = 4      # requests in parallel; lower it if the API often reports rate limits
NLI_BATCH = 16

# Files
P_TEMPLATE, P_EXAMPLES = "protocol/coding_prompt_template.txt", "protocol/coding_examples.csv"
P_KEY, P_HAND, P_ITEMS = "protocol/check_set_key.csv", "protocol/check_set_hand_codes.csv", "protocol/check_set_items.csv"
P_FREEZE, P_PROMPT_OUT = "protocol/coding_freeze.json", "protocol/coding_system_prompt.txt"
P_BUNDLE = "protocol/coding_freeze_bundle.zip"
RAW1, RAW2 = "protocol/llm_raw/study1.jsonl", "protocol/llm_raw/study2.jsonl"
NLI1, NLI2 = "protocol/nli_raw/study1.csv", "protocol/nli_raw/study2.csv"
P_RESULTS, P_LLM_CHECK = "protocol/check_set_results.csv", "protocol/check_set_llm_codes.csv"
P_CODES, P_LOG = "data/processed/justification_codes.csv", "protocol/coding_run_log.json"

In [5]:
# Functions. Nothing here calls a model.
UNAWARE = ["Unaware, at fault", "Unaware, no fault"]
CANON = {c.casefold(): c for c in CODES}

def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
def sha256_text(s): return hashlib.sha256(s.encode("utf-8")).hexdigest()
def sha256_file(path):
    with open(path, "rb") as fh: return hashlib.sha256(fh.read()).hexdigest()
def clean(text): return str(text).strip()
def read_csv(path, **kw): return pd.read_csv(path, dtype=str, keep_default_na=False, **kw)

# The prompt: the template with the fourteen examples in codebook order.
examples = read_csv(P_EXAMPLES)
check_key = read_csv(P_KEY)
assert len(examples) == 14 and examples.code.value_counts().to_dict() == {c: 2 for c in CODES}
assert examples.merge(check_key, on=["session_id", "vignette_position"]).empty, "An example is in the check set."
examples = examples.assign(order=examples.code.map(CODES.index)).sort_values("order", kind="stable")
EXAMPLE_BLOCK = "\n\n".join(
    "<example>\n" + USER_TEMPLATE.format(name=r.defendant, text=clean(r.justification))
    + f"\n<code>{r.code}</code>\n</example>" for r in examples.itertuples())
template = open(P_TEMPLATE, encoding="utf-8").read()
assert template.count("<<EXAMPLES>>") == 1
SYSTEM_PROMPT = template.replace("<<EXAMPLES>>", EXAMPLE_BLOCK).strip()

def build_request(name, text):
    """The request for one justification. Only the defendant's name and the justification are sent."""
    return {"model": MODEL, "max_tokens": MAX_TOKENS, "temperature": TEMPERATURE,
            "system": [{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}],
            "messages": [{"role": "user", "content": USER_TEMPLATE.format(name=name, text=clean(text))}],
            "output_config": {"format": {"type": "json_schema", "schema": SCHEMA}}}

def request_sha256(req): return sha256_text(json.dumps(req, sort_keys=True, ensure_ascii=False))

def send(req):
    """anthropic 1.x has no temperature argument, so the registered temperature goes into the request body."""
    body = {k: v for k, v in req.items() if k != "temperature"}
    return client.messages.create(**body, extra_body={"temperature": req["temperature"]}).model_dump(mode="json")

def read_code(resp):
    """The code in a saved response, with the registered rule for unreadable responses. Returns (code, problem)."""
    if resp.get("stop_reason") != "end_turn":
        return "Uncodable", f"stop_reason {resp.get('stop_reason')}"
    text = "".join(b.get("text", "") for b in resp.get("content") or [] if b.get("type") == "text")
    try:
        value = json.loads(text)["code"]
    except (ValueError, KeyError, TypeError):
        return "Uncodable", "no code in the response"
    code = CANON.get(" ".join(str(value).split()).casefold())
    return (code, None) if code else ("Uncodable", "label outside the seven codes")

def load_raw(path):
    """Saved responses, keyed by (participant, vignette_position)."""
    out = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                if line.strip():
                    rec = json.loads(line)
                    k = (rec["participant"], int(rec["vignette_position"]))
                    assert k not in out, f"{path} holds two responses for {k}."
                    out[k] = rec
    return out

_write_lock = threading.Lock()

def code_items(items, path, label):
    """Sends each justification in items (participant, vignette_position, name, text) that is not yet saved in path."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    saved, todo = load_raw(path), []
    for r in items.itertuples(index=False):
        k, req = (r.participant, int(r.vignette_position)), build_request(r.name, r.text)
        if k in saved:
            assert saved[k]["request_sha256"] == request_sha256(req), (
                f"{label}: the saved response for {k} was made with a different prompt or text.")
        else:
            todo.append((k, req))
    print(f"{label}: {len(items)} justifications, {len(items) - len(todo)} already coded, {len(todo)} to send.")
    def work(job):
        k, req = job
        resp = send(req)
        code, problem = read_code(resp)
        rec = {"participant": k[0], "vignette_position": k[1], "request_sha256": request_sha256(req),
               "received_at": now(), "code": code, "problem": problem, "response": resp}
        with _write_lock, open(path, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
    if todo:
        pool, n = ThreadPoolExecutor(max_workers=WORKERS), 0
        try:
            for f in as_completed([pool.submit(work, j) for j in todo]):
                f.result()
                n += 1
                if n % 100 == 0 or n == len(todo):
                    print(f"  {n} of {len(todo)} sent")
        finally:
            pool.shutdown(wait=True, cancel_futures=True)

def cohen_kappa(a, b, labels):
    a, b = np.asarray(a), np.asarray(b)
    po = np.mean(a == b)
    pe = sum(np.mean(a == lab) * np.mean(b == lab) for lab in labels)
    return float("nan") if np.isclose(pe, 1) else float((po - pe) / (1 - pe))

# Inference model
_nli = {}
NLI_LABELS = ["entailment", "neutral", "contradiction"]
NLI_COLS = [f"h{j}_{lab}" for j in (1, 2) for lab in NLI_LABELS]

def nli_model():
    if not _nli:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tok = AutoTokenizer.from_pretrained(NLI_MODEL, revision=NLI_REVISION)
        mod = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL, revision=NLI_REVISION, use_safetensors=True)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        mod.to(dev).eval()
        index = {v.lower(): int(k) for k, v in mod.config.id2label.items()}
        assert set(index) == set(NLI_LABELS), mod.config.id2label
        _nli.update(tok=tok, mod=mod, dev=dev, order=[index[lab] for lab in NLI_LABELS])
    return _nli

def nli_inputs(name, text):
    return NLI_PREMISE.format(name=name, text=clean(text)).strip(), [h.format(name=name) for h in NLI_HYPOTHESES]

def nli_probs(premises, hypotheses):
    """Probabilities of entailment, neutral and contradiction, in that order."""
    m, out = nli_model(), []
    for i in range(0, len(premises), NLI_BATCH):
        enc = m["tok"](premises[i:i + NLI_BATCH], hypotheses[i:i + NLI_BATCH], truncation="only_first",
                       max_length=512, padding=True, return_tensors="pt").to(m["dev"])
        with torch.inference_mode():
            p = torch.softmax(m["mod"](**enc).logits.float(), dim=-1).cpu().numpy()
        out.append(p[:, m["order"]])
    return np.concatenate(out)

def nli_items(items, path, label):
    """Runs the inference model on each justification in items that is not yet saved in path."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    have = {}
    if os.path.exists(path):
        saved = read_csv(path)
        have = dict(zip(zip(saved.participant, saved.vignette_position.astype(int)), saved.input_sha256))
    todo = []
    for r in items.itertuples(index=False):
        k = (r.participant, int(r.vignette_position))
        prem, hyps = nli_inputs(r.name, r.text)
        h = sha256_text(json.dumps([NLI_MODEL, NLI_REVISION, prem, hyps], ensure_ascii=False))
        if k in have:
            assert have[k] == h, f"{label}: the saved inference result for {k} was made with different inputs."
        else:
            todo.append((k, prem, hyps, h))
    print(f"{label}: {len(items)} justifications, {len(items) - len(todo)} already checked, {len(todo)} to run"
          + (f" on {nli_model()['dev']}." if todo else "."))
    for start in range(0, len(todo), 200):          # saved every 200, so an interruption loses little
        chunk = todo[start:start + 200]
        rows = pd.DataFrame({"participant": [c[0][0] for c in chunk], "vignette_position": [c[0][1] for c in chunk],
                             "input_sha256": [c[3] for c in chunk], "device": nli_model()["dev"]})
        for j in (0, 1):
            p = nli_probs([c[1] for c in chunk], [c[2][j] for c in chunk])
            for col, lab in enumerate(NLI_LABELS):
                rows[f"h{j + 1}_{lab}"] = p[:, col]
        rows.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        print(f"  {start + len(chunk)} of {len(todo)} done")

def nli_denied(df):
    """The registered rule: denied when entailment is the most probable label for either hypothesis."""
    top = lambda j: df[[f"h{j}_{lab}" for lab in NLI_LABELS]].astype(float).to_numpy().argmax(axis=1) == 0
    return (top(1) | top(2)).astype(int)

# Data. Only the columns needed for coding are read.
def study1_items():
    s1 = read_csv("data/study1_trials.csv", usecols=["session_id", "vignette_position", "judgment", "justification"])
    s1 = s1[s1.groupby("session_id").session_id.transform("size") == 5]
    s1 = s1.merge(read_csv("data/study1_names.csv"), on=["session_id", "vignette_position"], how="left", validate="one_to_one")
    assert len(s1) == 1060 and s1.defendant.fillna("").ne("").all()
    return (s1.rename(columns={"session_id": "participant", "defendant": "name", "justification": "text"})
              [["participant", "vignette_position", "name", "text", "judgment"]].reset_index(drop=True))

def study2_items():
    path = "data/processed/study2_trials.csv"
    assert os.path.exists(path), (f"{path} is missing. Run 01a_exclusions.R and then 01b_assemble_tables.R; "
                                  "they create it without printing any outcome.")
    s2 = read_csv(path, usecols=["participant", "vignette_position", "defendant", "justification"])
    assert not s2.duplicated(["participant", "vignette_position"]).any() and s2.groupby("participant").size().eq(5).all()
    assert s2.defendant.ne("").all()
    return s2.rename(columns={"defendant": "name", "justification": "text"}).reset_index(drop=True)

def codes_table(study, items, raw_path, nli_path, rederive=False):
    """One row per justification: the language model's code and the inference model's flag."""
    raw = load_raw(raw_path)
    keys = list(zip(items.participant, items.vignette_position.astype(int)))
    missing = [k for k in keys if k not in raw]
    assert not missing, f"Study {study}: {len(missing)} justifications have no saved response. Run Part C."
    code = [read_code(raw[k]["response"])[0] if rederive else raw[k]["code"] for k in keys]
    nli = read_csv(nli_path)
    nli = pd.DataFrame({"participant": nli.participant, "vignette_position": nli.vignette_position.astype(int),
                        "nli_denied": nli_denied(nli)})
    out = pd.DataFrame({"study": study, "participant": [k[0] for k in keys], "vignette_position": [k[1] for k in keys],
                        "code": code})
    out = out.merge(nli, on=["participant", "vignette_position"], how="left", validate="one_to_one")
    assert out.nli_denied.notna().all(), f"Study {study}: inference results are missing. Run Part C."
    out["nli_denied"] = out.nli_denied.astype(int)
    return out

# The freeze
def protocol_record():
    return {"model": MODEL, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS, "output_schema": SCHEMA,
            "user_message_template": USER_TEMPLATE, "text_preparation": TEXT_PREP,
            "invalid_response_rule": INVALID_RULE, "system_prompt_sha256": sha256_text(SYSTEM_PROMPT),
            "prompt_template_sha256": sha256_file(P_TEMPLATE), "examples_sha256": sha256_file(P_EXAMPLES),
            "check_set_key_sha256": sha256_file(P_KEY), "hand_codes_sha256": sha256_file(P_HAND),
            "nli_model": NLI_MODEL, "nli_revision": NLI_REVISION, "nli_premise": NLI_PREMISE,
            "nli_hypotheses": NLI_HYPOTHESES, "nli_rule": NLI_RULE, "anthropic_sdk": ANTHROPIC_VERSION}

def require_freeze():
    assert os.path.exists(P_FREEZE), "The protocol is not frozen yet. Run Part A first."
    with open(P_FREEZE, encoding="utf-8") as fh:
        frozen = json.load(fh)
    current = protocol_record()
    changed = [k for k in current if frozen.get(k) != current[k]]
    if changed:
        raise RuntimeError(f"These parts of the protocol differ from the version frozen at {frozen['frozen_at']}: "
                           + ", ".join(changed) + ". Undo the change; the registration does not allow revising "
                           "the protocol after the freeze.")
    return frozen

print("Functions ready. The prompt has", len(SYSTEM_PROMPT.split()), "words.")

Functions ready. The prompt has 864 words.


## Part A. The prompt, a test request and the freeze

The prompt below is the system prompt for every request. It holds the registered codebook and rules 1 to 4 word for word (rules 1 to 4 are the rules on the hand-coding sheet), clarifications 5 to 9, and two examples per code. Each request then sends one justification and the defendant's name, in the same form as the examples.

In [6]:
print(SYSTEM_PROMPT)
print("\n--- Each request then sends ---\n" + USER_TEMPLATE.format(name="[name]", text="[justification]"))

You will code short written justifications from a study of how people judge a criminal defendant's mental state.

In every version of the case, a man agreed to bring a package back across the border into the United States for someone else, and border agents found illicit drugs inside it. Each participant read one version, judged the defendant's mental state, and then wrote a justification for that judgment. You see only the justification and the defendant's name. You do not see the participant's judgment or the details of the version they read.

Assign exactly one of the seven codes below, for what the justification says or implies about the defendant's mental state regarding the contents of the package.

Codebook

Knew: The text says or implies that he knew, or was practically certain, that the package held illicit material.
Suspected: The text says or implies that he was aware of a possibility or risk that it did.
Unaware, at fault: He did not know or suspect, but a careful person wo

In [7]:
def get_key():
    k = os.environ.get("MPC_Claude_Key")
    if not k and IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("MPC_Claude_Key")
        except Exception:
            k = None
    return (k or getpass.getpass("Anthropic API key (hidden, kept in memory only): ")).strip()

client = anthropic.Anthropic(api_key=get_key(), max_retries=8, timeout=120)
print("API client ready. The key is not shown or saved.")

API client ready. The key is not shown or saved.


In [8]:
# One test request with the first example (a Study 1 text outside the check set) confirms that the model
# accepts temperature 0 and structured output. It is saved, and skipped on later runs.
if os.path.exists(P_FREEZE):
    require_freeze()
ex = examples.iloc[0]
pre_path = f"protocol/llm_raw/test_request_{sha256_text(SYSTEM_PROMPT)[:12]}.jsonl"
try:
    code_items(pd.DataFrame([{"participant": "example", "vignette_position": 1,
                              "name": ex.defendant, "text": ex.justification}]), pre_path, "Test request")
except anthropic.BadRequestError as e:
    raise RuntimeError(f"The API refused the registered settings: {e.message}") from None
rec = load_raw(pre_path)[("example", 1)]
print(f"{rec['response']['model']} answered {rec['code']!r} for an example coded {ex.code!r} "
      f"(stop reason {rec['response']['stop_reason']}).")

Test request: 1 justifications, 0 already coded, 1 to send.
  1 of 1 sent
claude-opus-4-6 answered 'Knew' for an example coded 'Knew' (stop reason end_turn).


The freeze records the settings above and checksums of the prompt, the examples, the check-set key and the hand codes, with the time. Run it once, when the prompt is final. It also writes `protocol/coding_freeze_bundle.zip` (the freeze record, the full prompt, the template, the examples, the check-set key, items and hand codes, and this notebook, if it is saved in the folder). Post the bundle on OSF before Part B; the registration fixes the protocol before the check set is scored.

In [9]:
if not os.path.exists(P_FREEZE):
    record = {"frozen_at": now(), **protocol_record()}
    with open(P_PROMPT_OUT, "w", encoding="utf-8") as fh:
        fh.write(SYSTEM_PROMPT + "\n")
    with open(P_FREEZE, "w", encoding="utf-8") as fh:
        json.dump(record, fh, indent=2, ensure_ascii=False)
    with zipfile.ZipFile(P_BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
        for f in [P_FREEZE, P_PROMPT_OUT, P_TEMPLATE, P_EXAMPLES, P_KEY, P_ITEMS, P_HAND, "02b_coding_protocol.ipynb"]:
            if os.path.exists(f):
                z.write(f)
    print(f"Frozen at {record['frozen_at']}. Post {P_BUNDLE} on OSF, then run Part B.")
else:
    print("The protocol matches the freeze of", require_freeze()["frozen_at"] + ".")

Frozen at 2026-10-08T22:29:39+00:00. Post protocol/coding_freeze_bundle.zip on OSF, then run Part B.


## Part B. The check set

The language model codes the 100 check-set justifications once, with the frozen prompt. Accuracy is Cohen's kappa between its codes and the hand codes: for the knowledge-stated code (Knew against every other code) within the 60 Knowledge items, and for the awareness-denied code (either unaware code against every other code) across all 100, each with the registered threshold of .60, and for all seven codes across all 100. The results go to `protocol/check_set_results.csv`, which the analysis script reads, and the language model's check-set codes to `protocol/check_set_llm_codes.csv`. The prompt cannot be revised after this point. The first cell stops until you confirm that the freeze bundle is on OSF.

In [11]:
# Set this to True once protocol/coding_freeze_bundle.zip is posted on OSF, so that the posting
# is timestamped before the check set is scored.
BUNDLE_POSTED_ON_OSF = True

require_freeze()
assert BUNDLE_POSTED_ON_OSF or os.path.exists(P_RESULTS), (
    "Post protocol/coding_freeze_bundle.zip on OSF first, then set BUNDLE_POSTED_ON_OSF = True in this cell.")
study1 = study1_items()
check = check_key.merge(study1, left_on=["session_id", "vignette_position"],
                        right_on=["participant", "vignette_position"], validate="one_to_one")
assert len(check) == 100 and check.judgment.value_counts().to_dict() == {
    "Knowledge": 60, "Recklessness": 20, "Negligence": 10, "Blamelessness": 10}
code_items(check[["participant", "vignette_position", "name", "text"]], RAW1, "Check set")

Check set: 100 justifications, 0 already coded, 100 to send.
  100 of 100 sent


In [12]:
saved = load_raw(RAW1)
check["llm_code"] = [saved[(p, int(v))]["code"] for p, v in zip(check.participant, check.vignette_position)]
hand = read_csv(P_HAND)
check = check.merge(hand[["item", "hand_code"]], on="item", validate="one_to_one")
check["hand_code"] = check.hand_code.str.strip()
assert check.hand_code.isin(CODES).all(), "A hand code is not one of the seven codes."

know = check[check.judgment == "Knowledge"]
k_stated = cohen_kappa(know.hand_code == "Knew", know.llm_code == "Knew", [False, True])
k_denied = cohen_kappa(check.hand_code.isin(UNAWARE), check.llm_code.isin(UNAWARE), [False, True])
k_seven = cohen_kappa(check.hand_code, check.llm_code, CODES)
met = lambda k: "reached" if k >= .60 else "not reached"
results = pd.DataFrame({
    "measure": ["knowledge_stated", "awareness_denied", "seven_codes"],
    "scope": ["60 Knowledge items", "all 100 items", "all 100 items"],
    "n_items": [len(know), len(check), len(check)],
    "kappa": [round(k, 4) for k in (k_stated, k_denied, k_seven)],
    "note": [f"Knew against every other code; registered threshold .60 {met(k_stated)}",
             f"Either unaware code against every other code; registered threshold .60 {met(k_denied)}",
             "All seven codes; reported without a threshold"]})
results.to_csv(P_RESULTS, index=False)
check[["item", "llm_code"]].sort_values("item").to_csv(P_LLM_CHECK, index=False)
display(results)
print("Check set: hand codes (rows) against the language model's codes (columns)")
display(pd.crosstab(pd.Categorical(check.hand_code, CODES), pd.Categorical(check.llm_code, CODES),
                    rownames=["hand"], colnames=["model"], dropna=False))

,measure,scope,n_items,kappa,note
0,knowledge_stated,60 Knowledge items,60,0.7826,Knew against every other code; registered thre...
1,awareness_denied,all 100 items,100,0.8344,Either unaware code against every other code; ...
2,seven_codes,all 100 items,100,0.8000,All seven codes; reported without a threshold


Check set: hand codes (rows) against the language model's codes (columns)


model,Knew,Suspected,"Unaware, at fault","Unaware, no fault",Act or situation only,Evidentiary,Uncodable
hand,,,,,,,
Knew,8,3,0,0,0,0,0
Suspected,2,25,3,0,1,0,0
"Unaware, at fault",0,0,16,0,0,0,0
"Unaware, no fault",0,0,0,20,0,0,0
Act or situation only,0,2,2,2,8,0,0
Evidentiary,0,0,0,0,0,1,0
Uncodable,0,0,1,0,0,0,6


## Part C. Coding every justification

The language model codes all 1,060 Study 1 justifications (the 100 check-set codes are reused, since the registration has them coded once) and every Study 2 justification. The inference model then checks every justification for an awareness denial. On a Colab GPU the inference model takes a few minutes; on a laptop processor it can take an hour or more, and it saves its progress every 200 justifications.

Nothing about the Study 2 codes is printed here.

In [13]:
require_freeze()
assert os.path.exists(P_RESULTS), "Score the check set (Part B) first."
study1, study2 = study1_items(), study2_items()
code_items(study1[["participant", "vignette_position", "name", "text"]], RAW1, "Study 1")
code_items(study2, RAW2, "Study 2")

Study 1: 1060 justifications, 100 already coded, 960 to send.
  100 of 960 sent
  200 of 960 sent
  300 of 960 sent
  400 of 960 sent
  500 of 960 sent
  600 of 960 sent
  700 of 960 sent
  800 of 960 sent
  900 of 960 sent
  960 of 960 sent
Study 2: 1310 justifications, 0 already coded, 1310 to send.
  100 of 1310 sent
  200 of 1310 sent
  300 of 1310 sent
  400 of 1310 sent
  500 of 1310 sent
  600 of 1310 sent
  700 of 1310 sent
  800 of 1310 sent
  900 of 1310 sent
  1000 of 1310 sent
  1100 of 1310 sent
  1200 of 1310 sent
  1300 of 1310 sent
  1310 of 1310 sent


In [14]:
nli_items(study1, NLI1, "Study 1")
nli_items(study2, NLI2, "Study 2")

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  870MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Study 1: 1060 justifications, 0 already checked, 1060 to run on cuda.
  200 of 1060 done
  400 of 1060 done
  600 of 1060 done
  800 of 1060 done
  1000 of 1060 done
  1060 of 1060 done
Study 2: 1310 justifications, 0 already checked, 1310 to run on cuda.
  200 of 1310 done
  400 of 1310 done
  600 of 1310 done
  800 of 1310 done
  1000 of 1310 done
  1200 of 1310 done
  1310 of 1310 done


In [15]:
codes = pd.concat([codes_table(1, study1, RAW1, NLI1), codes_table(2, study2, RAW2, NLI2)], ignore_index=True)
assert codes.code.isin(CODES).all() and not codes.duplicated(["study", "participant", "vignette_position"]).any()
os.makedirs(os.path.dirname(P_CODES), exist_ok=True)
codes.to_csv(P_CODES, index=False)

raws = {"study1": load_raw(RAW1), "study2": load_raw(RAW2)}
models = sorted({r["response"]["model"] for raw in raws.values() for r in raw.values()})
unreadable = {s: sum(r["problem"] is not None for r in raw.values()) for s, raw in raws.items()}
usage = {}
for raw in raws.values():
    for r in raw.values():
        for k, v in (r["response"].get("usage") or {}).items():
            if isinstance(v, (int, float)):
                usage[k] = usage.get(k, 0) + v
check_times = [raws["study1"][(p, int(v))]["received_at"] for p, v in zip(check_key.session_id, check_key.vignette_position)]
log = {"written_at": now(), "protocol_frozen_at": require_freeze()["frozen_at"],
       "check_set_coded": [min(check_times), max(check_times)],
       "justifications": {"study1": int((codes.study == 1).sum()), "study2": int((codes.study == 2).sum())},
       "responses_coded_uncodable_because_unreadable": unreadable, "models_reported_by_api": models,
       "token_usage": usage, "nli_devices": sorted(set(read_csv(NLI1).device) | set(read_csv(NLI2).device)),
       "python": platform.python_version(), "platform": platform.platform(), "anthropic": anthropic.__version__,
       "torch": torch.__version__, "transformers": transformers.__version__, "codes_sha256": sha256_file(P_CODES)}
with open(P_LOG, "w", encoding="utf-8") as fh:
    json.dump(log, fh, indent=2)
print(f"Wrote {P_CODES}: {log['justifications']['study1']} Study 1 and {log['justifications']['study2']} Study 2 justifications.")
print("Responses coded Uncodable because they could not be read:", unreadable)
print("Model reported by the API:", ", ".join(models), "" if models == [MODEL] else f"(expected {MODEL})")

Wrote data/processed/justification_codes.csv: 1060 Study 1 and 1310 Study 2 justifications.
Responses coded Uncodable because they could not be read: {'study1': 0, 'study2': 0}
Model reported by the API: claude-opus-4-6 


## Part D. Rebuilding the codes from the archive

This part needs no key and no model. It reads every code again from the saved raw responses, recomputes every cross-check flag from the saved probabilities with the registered rule, and compares the result with `data/processed/justification_codes.csv`. Anyone with the archive can run the setup, the settings, the functions and this part.

In [16]:
rebuilt = pd.concat([codes_table(1, study1_items(), RAW1, NLI1, rederive=True),
                     codes_table(2, study2_items(), RAW2, NLI2, rederive=True)], ignore_index=True)
written = read_csv(P_CODES)
assert rebuilt.astype(str).equals(written.astype(str)), "The codes file does not match the archive."
print(f"The codes file matches the archive exactly (sha256 {sha256_file(P_CODES)[:16]}...).")

The codes file matches the archive exactly (sha256 f68c4ea99166895c...).
